# Neuro-CXG: Research-Grade Analysis Notebook

This notebook provides interactive exploration of the trained GNN model for ASD classification.

## Setup

In [4]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.core.config import *
from src.models.causal_gnn import CausalBrainGNN
from src.features.graph_factory import ABIDECausalDataset
from src.analysis.feature_importance import FeatureAttributionAnalyzer
from src.analysis.gradients.training_monitor import TrainingMonitor

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

print("✓ Imports successful")
print(f"Device: {DEVICE}")

✓ Imports successful
Device: cuda



## 1. Feature Attribution Analysis

### 1.1 Load Model and Data

In [8]:
# Feature names (14 total)
feature_names = [
    # Temporal features (8)
    'mean', 'std', 'skew', 'kurt', 'psd', 'mssd', 'range', 'autocorr',
    # Spatial features (6)
    'x', 'y', 'z_depth', 'size', 'conf_std', 'detection_count'
]

# Load test dataset
test_dataset = ABIDECausalDataset(split='test')
test_data = [test_dataset[i] for i in range(len(test_dataset)) if test_dataset[i] is not None]

from torch_geometric.loader import DataLoader
test_loader = DataLoader(test_data, batch_size=1)

print(f"Test set: {len(test_data)} subjects")

# Load best model (fold 0)
model = CausalBrainGNN(
    num_node_features=GNN_IN_CHANNELS,
    hidden_channels=GNN_HIDDEN_CHANNELS_TUNED,
    num_classes=2
).to(DEVICE)

checkpoint_path = CHECKPOINT_DIR / "best_model_fold0.pt"
checkpoint = torch.load(checkpoint_path, weights_only=False)
model.load_state_dict(checkpoint['model_state'])
model.eval()

print(f"✓ Loaded model from {checkpoint_path}")
print(f"  Best AUC: {checkpoint.get('auc', 'N/A'):.4f}")


INFO:src.features.graph_factory:✓ Feature dimensions validated
INFO:src.features.graph_factory:Initialized test dataset with 156 subjects
INFO:src.features.graph_factory:  Node features: 14 (8 temporal + 6 spatial)


Test set: 156 subjects
✓ Loaded model from /home/nidszxh/Projects/Neuro-CXG/models/checkpoints/best_model_fold0.pt
  Best AUC: 0.5971


### 1.2 Compute Feature Attributions

In [12]:
# Reload modules to get latest changes
import importlib
import src.analysis.feature_importance
importlib.reload(src.analysis.feature_importance)
from src.analysis.feature_importance import FeatureAttributionAnalyzer

# Initialize analyzer
analyzer = FeatureAttributionAnalyzer(
    model=model,
    test_loader=test_loader,
    feature_names=feature_names,
    device=DEVICE
)

# Compute attributions (this may take a few minutes)
print("Computing feature attributions...")
print("Note: This uses Integrated Gradients which requires the model to be differentiable.")

try:
    attributions = analyzer.compute_attributions(n_steps=50, debug=True)
    print(f"✓ Attribution shape: {attributions.shape}")
    print(f"  (num_samples, num_lobes, num_features)")
except Exception as e:
    print(f"✗ Error computing attributions: {e}")
    print("\nFallback: Using simplified analysis without attributions...")
    attributions = None

INFO:src.analysis.feature_importance:FeatureAttributionAnalyzer initialized
INFO:src.analysis.feature_importance:  Device: cuda
INFO:src.analysis.feature_importance:  Features: 14
INFO:src.analysis.feature_importance:Computing feature attributions...
INFO:src.analysis.feature_importance:  Integration steps: 50


Computing feature attributions...
Note: This uses Integrated Gradients which requires the model to be differentiable.


Computing attributions: 100%|██████████| 156/156 [00:00<00:00, 193.74it/s]
INFO:src.analysis.feature_importance:Successfully computed 0 batch attributions, 156 failed
ERROR:src.analysis.feature_importance:✗ No attributions computed! All 156 batches failed.
ERROR:src.analysis.feature_importance:  Check that model predictions are working and data has correct format.


✗ Error computing attributions: No attributions could be computed for any batch in the test set. This may indicate that the model forward pass is failing. Check the error messages above for details.

Fallback: Using simplified analysis without attributions...


### 1.3 Visualize Global Feature Importance

In [ ]:
# Create heatmap
output_dir = Path("results/analysis/feature_attribution")
output_dir.mkdir(parents=True, exist_ok=True)

mean_attr = analyzer.visualize_feature_importance(
    attributions,
    output_dir / "feature_importance_heatmap.png"
)

# Display inline
from IPython.display import Image, display
display(Image(filename=str(output_dir / "feature_importance_heatmap.png")))

### 1.4 Temporal vs Spatial Feature Comparison

In [ ]:
# Statistical comparison
results = analyzer.compare_temporal_vs_spatial(attributions)

# Visualize as bar chart
fig, ax = plt.subplots(figsize=(10, 6))

categories = ['Temporal\n(fMRI stats)', 'Spatial\n(YOLO coords)']
values = [results['temporal_mean'], results['spatial_mean']]
colors = ['#3498db', '#e74c3c']

bars = ax.bar(categories, values, color=colors, alpha=0.7, edgecolor='black', linewidth=2)

# Add value labels
for bar, value in zip(bars, values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
           f'{value:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

# Add significance annotation
if results['significant']:
    ax.text(0.5, max(values) * 1.1, 
           f'p = {results["p_value"]:.4f} *', 
           ha='center', fontsize=14, fontweight='bold', color='green')
else:
    ax.text(0.5, max(values) * 1.1, 
           f'p = {results["p_value"]:.4f} (n.s.)', 
           ha='center', fontsize=12, color='gray')

ax.set_ylabel('Mean Attribution Magnitude', fontsize=14, fontweight='bold')
ax.set_title('Temporal vs Spatial Feature Importance', fontsize=16, fontweight='bold', pad=20)
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(output_dir / 'temporal_vs_spatial.png', dpi=300, bbox_inches='tight')
plt.show()

### 1.5 Per-Lobe Feature Analysis

In [ ]:
# Analyze each lobe separately
lobe_results = analyzer.analyze_per_lobe(attributions, output_dir / "per_lobe")

# Visualize top features per lobe
lobe_names = ['Frontal', 'Temporal', 'Parietal', 'Occipital', 'Limbic']

fig, axes = plt.subplots(1, 5, figsize=(20, 5))

for idx, (lobe_name, ax) in enumerate(zip(lobe_names, axes)):
    lobe_data = lobe_results[lobe_results['lobe'] == lobe_name].head(8)
    
    bars = ax.barh(lobe_data['feature'], lobe_data['importance'], 
                   color='#3498db', alpha=0.7, edgecolor='black')
    
    ax.set_xlabel('Importance', fontsize=10)
    ax.set_title(lobe_name, fontsize=12, fontweight='bold')
    ax.invert_yaxis()
    ax.grid(alpha=0.3, axis='x')

plt.suptitle('Top 8 Features per Brain Lobe', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(output_dir / 'per_lobe_top_features.png', dpi=300, bbox_inches='tight')
plt.show()

### 1.6 ASD vs Control Comparison

In [ ]:
# Compare feature importance between classes
analyzer.visualize_per_class(
    attributions,
    analyzer.labels,
    output_dir / "per_class_comparison.png"
)

display(Image(filename=str(output_dir / "per_class_comparison.png")))


## 2. Training Dynamics Analysis

### 2.1 Load Training History

In [ ]:
# Initialize monitor
monitor = TrainingMonitor(output_dir='results/analysis/training', num_folds=5)

# Load saved training history for each fold
for fold_id in range(5):
    history_path = Path(f'results/analysis/training/training_history_fold_{fold_id}.json')
    if history_path.exists():
        monitor.load_history(fold_id, history_path)
        print(f"✓ Loaded history for fold {fold_id}")
    else:
        print(f"⚠ History not found for fold {fold_id}")

### 2.2 Visualize Training Curves

In [ ]:
# Generate training curve plots for all folds
for fold_id in range(5):
    if monitor.fold_histories[fold_id]['train_loss']:
        path = monitor.plot_training_curves(fold_id)
        print(f"✓ Saved training curves for fold {fold_id}")

### 2.3 Analyze Learning Patterns

In [ ]:
# Extract key metrics from all folds
fold_summaries = []

for fold_id in range(5):
    history = monitor.fold_histories[fold_id]
    
    if not history['val_auc']:
        continue
    
    best_auc = max(history['val_auc'])
    best_epoch = history['val_auc'].index(best_auc) + 1
    final_auc = history['val_auc'][-1]
    
    # Check for overfitting
    overfit_gap = history['train_loss'][-1] - history['val_loss'][-1]
    
    fold_summaries.append({
        'Fold': fold_id,
        'Best AUC': best_auc,
        'Final AUC': final_auc,
        'Best Epoch': best_epoch,
        'Overfit Gap': overfit_gap
    })

summary_df = pd.DataFrame(fold_summaries)
print("\n" + "="*70)
print("TRAINING SUMMARY ACROSS FOLDS")
print("="*70)
print(summary_df.to_string(index=False))
print("="*70)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Best AUC distribution
ax = axes[0]
ax.hist(summary_df['Best AUC'], bins=10, color='#3498db', alpha=0.7, edgecolor='black')
ax.axvline(summary_df['Best AUC'].mean(), color='red', linestyle='--', 
          linewidth=2, label=f'Mean: {summary_df["Best AUC"].mean():.4f}')
ax.set_xlabel('Best Validation AUC', fontsize=12, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax.set_title('Distribution of Best AUC Across Folds', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# Convergence speed
ax = axes[1]
ax.scatter(summary_df['Best Epoch'], summary_df['Best AUC'], 
          s=200, c='#e74c3c', alpha=0.7, edgecolor='black', linewidths=2)
for idx, row in summary_df.iterrows():
    ax.annotate(f'Fold {row["Fold"]}', 
               (row['Best Epoch'], row['Best AUC']),
               xytext=(5, 5), textcoords='offset points', fontsize=10)
ax.set_xlabel('Best Epoch', fontsize=12, fontweight='bold')
ax.set_ylabel('Best AUC', fontsize=12, fontweight='bold')
ax.set_title('Convergence Speed vs Performance', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('results/analysis/training/convergence_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

### 2.4 Confusion Matrix Evolution

In [ ]:
# Plot confusion matrix evolution for best fold
best_fold = summary_df.loc[summary_df['Best AUC'].idxmax(), 'Fold']
print(f"Best fold: {best_fold} (AUC: {summary_df.loc[summary_df['Best AUC'].idxmax(), 'Best AUC']:.4f})")

monitor.plot_confusion_evolution(best_fold)

## 3. Graph Structure Analysis

### 3.1 Load Causal Graphs

In [ ]:
# Load all causal graphs
graph_dir = CAUSAL_GRAPHS_DIR

graph_files = list(graph_dir.glob("*_graph.pt"))
print(f"Found {len(graph_files)} causal graphs")

# Load manifest for labels
manifest = pd.read_csv(MASTER_MANIFEST)
manifest['subject_id'] = manifest['subject_id'].astype(str)

### 3.2 Compute Graph Properties

In [ ]:
import networkx as nx

graph_properties = []

# Sample 200 graphs for analysis (faster)
sample_files = np.random.choice(graph_files, min(200, len(graph_files)), replace=False)

for graph_file in sample_files:
    # Load graph
    graph_data = torch.load(graph_file)
    subject_id = graph_file.stem.replace('_graph', '')
    
    # Get label
    sub_manifest = manifest[manifest['subject_id'] == subject_id]
    if len(sub_manifest) == 0:
        continue
    dx_group = sub_manifest.iloc[0]['DX_GROUP']
    
    # Convert to NetworkX
    adj = graph_data['adj'].numpy()
    G = nx.DiGraph(adj)
    
    # Compute properties
    num_edges = G.number_of_edges()
    density = nx.density(G)
    
    # Degree centrality
    in_deg = dict(G.in_degree())
    out_deg = dict(G.out_degree())
    
    graph_properties.append({
        'subject_id': subject_id,
        'dx_group': dx_group,
        'num_edges': num_edges,
        'density': density,
        'frontal_in': in_deg[0],
        'frontal_out': out_deg[0],
        'limbic_in': in_deg[4],
        'limbic_out': out_deg[4]
    })

props_df = pd.DataFrame(graph_properties)
print(f"Analyzed {len(props_df)} graphs")

### 3.3 Compare ASD vs Control Graph Topology

In [ ]:
# Separate by diagnosis
asd_graphs = props_df[props_df['dx_group'] == 1]
control_graphs = props_df[props_df['dx_group'] == 2]

# Statistical comparison
from scipy.stats import mannwhitneyu

metrics = ['num_edges', 'density', 'frontal_in', 'limbic_out']

print("\n" + "="*70)
print("GRAPH TOPOLOGY: ASD vs CONTROL")
print("="*70)

for metric in metrics:
    asd_vals = asd_graphs[metric].values
    control_vals = control_graphs[metric].values
    
    u_stat, p_value = mannwhitneyu(asd_vals, control_vals, alternative='two-sided')
    
    print(f"\n{metric}:")
    print(f"  ASD:     {asd_vals.mean():.4f} ± {asd_vals.std():.4f}")
    print(f"  Control: {control_vals.mean():.4f} ± {control_vals.std():.4f}")
    print(f"  Mann-Whitney U: {u_stat:.2f}, p={p_value:.4f}")
    
    if p_value < 0.05:
        direction = "higher" if asd_vals.mean() > control_vals.mean() else "lower"
        print(f"  ✓ ASD has significantly {direction} {metric}")
    else:
        print(f"  → No significant difference")

print("="*70)

### 3.4 Visualize Graph Topology Differences

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for idx, metric in enumerate(metrics):
    ax = axes[idx // 2, idx % 2]
    
    # Box plot
    data = [asd_graphs[metric].values, control_graphs[metric].values]
    bp = ax.boxplot(data, labels=['ASD', 'Control'], patch_artist=True)
    
    # Color boxes
    bp['boxes'][0].set_facecolor('#e74c3c')
    bp['boxes'][0].set_alpha(0.7)
    bp['boxes'][1].set_facecolor('#3498db')
    bp['boxes'][1].set_alpha(0.7)
    
    ax.set_ylabel(metric.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    ax.set_title(f'{metric.replace("_", " ").title()} Distribution', 
                fontsize=14, fontweight='bold')
    ax.grid(alpha=0.3, axis='y')

plt.suptitle('Graph Topology: ASD vs Control', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('results/analysis/graph_topology_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

### 3.5 Average Causal Graphs

In [ ]:
# Compute average adjacency matrix per group
asd_adjs = []
control_adjs = []

for graph_file in sample_files:
    graph_data = torch.load(graph_file)
    subject_id = graph_file.stem.replace('_graph', '')
    
    sub_manifest = manifest[manifest['subject_id'] == subject_id]
    if len(sub_manifest) == 0:
        continue
    
    dx_group = sub_manifest.iloc[0]['DX_GROUP']
    adj = graph_data['adj'].numpy()
    
    if dx_group == 1:
        asd_adjs.append(adj)
    else:
        control_adjs.append(adj)

avg_asd = np.mean(asd_adjs, axis=0)
avg_control = np.mean(control_adjs, axis=0)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

lobe_names = ['Frontal', 'Temporal', 'Parietal', 'Occipital', 'Limbic']

# ASD
sns.heatmap(avg_asd, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
           xticklabels=lobe_names, yticklabels=lobe_names,
           ax=axes[0], vmin=-0.5, vmax=0.5, 
           cbar_kws={'label': 'Correlation'}, linewidths=0.5)
axes[0].set_title('ASD Average Causal Graph', fontsize=14, fontweight='bold')

# Control
sns.heatmap(avg_control, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
           xticklabels=lobe_names, yticklabels=lobe_names,
           ax=axes[1], vmin=-0.5, vmax=0.5, 
           cbar_kws={'label': 'Correlation'}, linewidths=0.5)
axes[1].set_title('Control Average Causal Graph', fontsize=14, fontweight='bold')

# Difference
diff = avg_asd - avg_control
sns.heatmap(diff, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
           xticklabels=lobe_names, yticklabels=lobe_names,
           ax=axes[2], vmin=-0.3, vmax=0.3, 
           cbar_kws={'label': 'Difference'}, linewidths=0.5)
axes[2].set_title('Difference (ASD - Control)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('results/analysis/average_causal_graphs.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Key Findings Summary

In [ ]:
print("\n" + "="*70)
print("NEURO-CXG: KEY RESEARCH FINDINGS")
print("="*70)

print("\n1. FEATURE IMPORTANCE:")
print(f"   - Temporal/Spatial ratio: {results['ratio']:.2f}")
if results['significant']:
    winner = "Temporal" if results['temporal_mean'] > results['spatial_mean'] else "Spatial"
    print(f"   - {winner} features significantly more important (p={results['p_value']:.4f})")
else:
    print(f"   - Both temporal and spatial contribute (p={results['p_value']:.4f})")

print("\n2. MODEL PERFORMANCE:")
print(f"   - Mean Best AUC: {summary_df['Best AUC'].mean():.4f} ± {summary_df['Best AUC'].std():.4f}")
print(f"   - Best single fold: {summary_df['Best AUC'].max():.4f}")
print(f"   - Average convergence: {summary_df['Best Epoch'].mean():.1f} epochs")

print("\n3. GRAPH TOPOLOGY (ASD vs Control):")
for metric in ['num_edges', 'density']:
    asd_mean = asd_graphs[metric].mean()
    control_mean = control_graphs[metric].mean()
    diff_pct = ((asd_mean - control_mean) / control_mean) * 100
    print(f"   - {metric}: ASD {diff_pct:+.1f}% vs Control")

print("="*70 + "\n")


# Save this analysis
print("✓ Analysis complete")
print("  All figures saved to results/analysis/")